# Flood forecasting with CNN-LSTM: model, results and explanations
**DL Lab project** · Anhad Mahajan & Aman Sumeet Saxena

This notebook builds and tests the models on all four regions:

1. **Data**: load the four floods and turn them into model inputs
2. **Model A, the CNN-LSTM forecaster**: predicts the water-depth map 30 minutes ahead
3. **Results of Model A**: compared with the simplest possible forecast, "copy the last map", plus the water-balance rule
4. **The trend forecast and Model B**: a second simple forecast, and a version of the network that starts from it
5. **Capsule network**: predicts how severe the flooding will be in 30 minutes
6. **Grad-CAM**: heatmaps of which areas the models look at
7. **NeuroSymbolic rules**: written rules that check the capsule network's answer and explain the decision
8. **Where we stand**: one summary table

Two models are trained, one per resolution:

| Model | Trained on | Tested on |
|---|---|---|
| 30 m | Australia (first 70% of the flood) | Australia (last 15%) and the **UK (never seen)** |
| 480 m | Pakistan (first 70% of the flood) | Pakistan (last 15%) and **Mozambique (never seen)** |

**Runtime:** GPU (T4 or L4), with High-RAM if available. The saved models are loaded from Drive, so nothing is retrained
and the numbers match the report. The whole notebook takes about 20–30 minutes.

## 0. Setup
Needs the prepared data from the *Data_preparation* notebook in `Flood_project/prepared_data/` on Google Drive.

In [ ]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from scipy.ndimage import uniform_filter
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
import keras
from keras import layers

from google.colab import drive

In [ ]:
# Check that a GPU is connected
gpus = tf.config.list_physical_devices("GPU")

if len(gpus) > 0:
    print("GPU found:", gpus[0].name)
else:
    print("No GPU found -> Runtime > Change runtime type > T4 GPU")

In [ ]:
# Connect Google Drive
drive.mount("/content/drive")

# Folders used by this notebook
PROJECT_DIR = "/content/drive/MyDrive/Flood_project"
PREPARED_DIR = os.path.join(PROJECT_DIR, "prepared_data")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
FIGURE_DIR = os.path.join(PROJECT_DIR, "model_outputs", "figures")
TABLE_DIR = os.path.join(PROJECT_DIR, "model_outputs", "tables")

# Create the output folders if they don't exist yet
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

print("Model weights will be saved in:", MODEL_DIR)
print("Figures will be saved in:      ", FIGURE_DIR)
print("Tables will be saved in:       ", TABLE_DIR)

In [ ]:
# The two models: each one is trained on one region and tested on another region it has never seen
PAIRS = [
    {"name": "30m", "train": "Australia_30m", "test": "UK_30m"},
    {"name": "480m", "train": "Pakistan_480m", "test": "Mozambique_480m"},
]

REGION_NAMES = {
    "Australia_30m": "Australia (30 m)",
    "UK_30m": "UK (30 m)",
    "Pakistan_480m": "Pakistan (480 m)",
    "Mozambique_480m": "Mozambique (480 m)",
}

# Time between two maps
STEP_HOURS = 0.5

# A cell counts as "wet" above 1 cm of water, and as "flooded" above 30 cm
WET_DEPTH_M = 0.01
FLOOD_DEPTH_M = 0.30

# Severity label: share of a tile's normally-dry land under more than 30 cm of water
SEVERITY_LIMITS = [0.05, 0.15, 0.30]
SEVERITY_NAMES = ["None", "Minor", "Moderate", "Severe"]

# Time split inside the training region: first 70% train, next 15% validation, last 15% test
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15

# The model sees the last 4 maps (2 hours) and predicts the next map (30 minutes ahead)
INPUT_STEPS = 4

# Each past map gives the model 3 layers: water depth, rain that already fell, local ground height
CHANNELS = 3

# Training uses square tiles of 64 x 64 cells; testing uses whole maps
TILE_SIZE = 64
TRAIN_TILES = 4000
VALIDATION_TILES = 600
TEST_TILES = 1000

# Keep every tile, calm ones included. Keeping only some calm tiles taught the model
# that water always rises, so it added water everywhere when nothing was happening.
CHANGE_LIMIT_M = 0.01
CALM_TILE_SHARE = 1.0

# Rain is divided by 10 so its values are around the same size as the other inputs (mm per hour / 10)
RAIN_SCALE = 10.0

# Local ground height = ground height minus the average of the surrounding 15 x 15 cells
TERRAIN_WINDOW = 15

# Training settings
EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 1e-3

# True: train new models. False: load the models saved on Drive (it still trains if nothing is saved yet)
TRAIN_NEW = False
TRAIN_V2 = False

# Same random numbers every run, so results can be repeated
keras.utils.set_random_seed(42)

# Blue colour scale for water; dry cells are shown in grey
water_colours = ListedColormap(plt.cm.Blues(np.linspace(0.3, 1.0, 256)))
water_colours.set_bad("lightgrey")

In [ ]:
def save_figure(fig, name):
    """Saves a figure as a PNG file in the figures folder on Drive."""

    path = os.path.join(FIGURE_DIR, name + ".png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved figure:", name + ".png")


def save_table(table, name):
    """Saves a table as a CSV file in the tables folder on Drive."""

    path = os.path.join(TABLE_DIR, name + ".csv")
    table.to_csv(path, index=False)
    print("Saved table:", name + ".csv")


def show_water(ax, depth_map, top_value):
    """Draws a water-depth map: dry cells in grey, water in blue."""

    water = np.array(depth_map, dtype="float32")
    water[water < WET_DEPTH_M] = np.nan

    return ax.imshow(water, cmap=water_colours, vmin=0, vmax=top_value)


def flooded_share(depth_map, dry_land):
    """Share of the normally-dry land that is under more than 30 cm of water."""

    if dry_land.sum() == 0:
        return 0.0

    return float(np.mean(depth_map[dry_land] > FLOOD_DEPTH_M))


def severity_class(share):
    """Turns a flooded share into a class: 0 None, 1 Minor, 2 Moderate, 3 Severe."""

    return int(np.digitize(share, SEVERITY_LIMITS))


def cut_into_tiles(rows, columns):
    """Returns the top-left corner of every full 64 x 64 tile in a map."""

    corners = []

    for top in range(0, rows - TILE_SIZE + 1, TILE_SIZE):
        for left in range(0, columns - TILE_SIZE + 1, TILE_SIZE):
            corners.append((top, left))

    return corners

## 1. Data
For every 30-minute step the model gets three layers:

| Layer | What it is | Why the model needs it |
|---|---|---|
| Water depth | metres of water in each cell | the thing we forecast |
| Rain | rain that fell in the **previous** 30 minutes (mm per hour / 10) | rain adds water; only rain that has already fallen is used, so the forecast is honest |
| Local ground height | how much higher or lower a cell is than its surroundings | water flows from high to low cells |

Local ground height is used instead of the raw height above sea level, because what moves water is the
*difference* between neighbouring cells, not the absolute height. It also keeps the 30 m and 480 m maps on
a similar scale.

In [ ]:
def load_region(dataset):
    """Loads one prepared dataset from Drive and adds the extra layers the model needs."""

    folder = os.path.join(PREPARED_DIR, dataset)

    # Water depth and rain for every 30-minute step (missing values become 0)
    depth = np.nan_to_num(np.load(os.path.join(folder, "depth.npy")), copy=False)
    rain = np.nan_to_num(np.load(os.path.join(folder, "rain.npy")), copy=False)

    # Ground height: fill missing cells with the average height
    terrain = np.load(os.path.join(folder, "terrain.npy")).astype("float32")
    terrain[np.isnan(terrain)] = np.nanmean(terrain)

    # Local ground height: positive = higher than the surroundings, negative = a dip or valley
    local_height = terrain - uniform_filter(terrain, size=TERRAIN_WINDOW)
    local_height = local_height / max(local_height.std(), 1e-6)

    # Rivers and lakes, so the severity label only counts land that is normally dry
    permanent_water = np.load(os.path.join(folder, "permanent_water.npy")).astype(bool)

    with open(os.path.join(folder, "map_info.json")) as file:
        info = json.load(file)

    number_of_maps = depth.shape[0]

    return {
        "depth": depth,
        "rain": rain,
        "local_height": local_height.astype("float32"),
        "dry_land": ~permanent_water,
        "info": info,
        "number_of_maps": number_of_maps,
        "train_end": int(TRAIN_SHARE * number_of_maps),
        "validation_end": int((TRAIN_SHARE + VALIDATION_SHARE) * number_of_maps),
    }

In [ ]:
# Load the four regions (takes a few minutes: Australia alone is about 3 GB)
regions = {}

for dataset in REGION_NAMES:
    regions[dataset] = load_region(dataset)
    print(dataset, "- loaded", regions[dataset]["depth"].shape)

In [ ]:
# The first map the model can forecast: it needs 4 past maps, plus the rain before the first of them
FIRST_TARGET = INPUT_STEPS + 1

# Which maps are used for what
split_rows = []

for pair in PAIRS:
    train_region = regions[pair["train"]]
    test_region = regions[pair["test"]]

    split_rows.append({
        "Model": pair["name"],
        "Region": REGION_NAMES[pair["train"]],
        "Train (hours)": str(FIRST_TARGET * STEP_HOURS) + " – " + str(train_region["train_end"] * STEP_HOURS),
        "Validation (hours)": str(train_region["train_end"] * STEP_HOURS) + " – " + str(train_region["validation_end"] * STEP_HOURS),
        "Test (hours)": str(train_region["validation_end"] * STEP_HOURS) + " – " + str(train_region["number_of_maps"] * STEP_HOURS),
    })

    split_rows.append({
        "Model": pair["name"],
        "Region": REGION_NAMES[pair["test"]],
        "Train (hours)": "-",
        "Validation (hours)": "-",
        "Test (hours)": "whole flood (never seen)",
    })

split_table = pd.DataFrame(split_rows)
display(split_table)
save_table(split_table, "01_data_split")

In [ ]:
def make_input(region, last_step, top=0, left=0, height=None, width=None):
    """Builds one model input: the 4 maps up to 'last_step', each with depth, past rain and local ground height.
    Without a tile position, the whole map is used. Shape: (4, height, width, 3)."""

    if height is None:
        height = region["depth"].shape[1]
        width = region["depth"].shape[2]

    rows = slice(top, top + height)
    columns = slice(left, left + width)

    steps = []

    for step in range(last_step - INPUT_STEPS + 1, last_step + 1):
        depth = region["depth"][step, rows, columns]

        # rain[step - 1] is the rain that fell between the previous map and this one, so it has already happened
        rain = region["rain"][step - 1, rows, columns].astype("float32") / RAIN_SCALE

        height_map = region["local_height"][rows, columns]

        steps.append(np.stack([depth, rain, height_map], axis=-1))

    return np.stack(steps).astype("float32")


def make_training_tiles(region, first_target, last_target, count):
    """Cuts random 64 x 64 tiles at random times. The target is the depth map 30 minutes after the last input."""

    rows = region["depth"].shape[1]
    columns = region["depth"].shape[2]

    inputs = []
    targets = []
    positions = []

    while len(inputs) < count:
        target_step = np.random.randint(first_target, last_target)
        top = np.random.randint(0, rows - TILE_SIZE + 1)
        left = np.random.randint(0, columns - TILE_SIZE + 1)

        target = region["depth"][target_step, top:top + TILE_SIZE, left:left + TILE_SIZE]
        last = region["depth"][target_step - 1, top:top + TILE_SIZE, left:left + TILE_SIZE]

        # Optionally skip calm tiles (with CALM_TILE_SHARE = 1.0 every tile is kept)
        water_changes = np.abs(target - last).max() > CHANGE_LIMIT_M

        if not water_changes and np.random.rand() > CALM_TILE_SHARE:
            continue

        inputs.append(make_input(region, target_step - 1, top, left, TILE_SIZE, TILE_SIZE))
        targets.append(target[:, :, None])
        positions.append((target_step, top, left))

    return np.array(inputs), np.array(targets, dtype="float32"), positions


def make_test_tiles(region, first_target, last_target, max_count):
    """Cuts maps into a regular grid of 64 x 64 tiles and picks up to 'max_count' of them at random."""

    rows = region["depth"].shape[1]
    columns = region["depth"].shape[2]

    all_positions = []

    for target_step in range(first_target, last_target):
        for top, left in cut_into_tiles(rows, columns):
            all_positions.append((target_step, top, left))

    picked = np.random.permutation(len(all_positions))[:max_count]
    positions = [all_positions[i] for i in picked]

    inputs = []
    targets = []

    for target_step, top, left in positions:
        inputs.append(make_input(region, target_step - 1, top, left, TILE_SIZE, TILE_SIZE))
        targets.append(region["depth"][target_step, top:top + TILE_SIZE, left:left + TILE_SIZE, None])

    return np.array(inputs), np.array(targets, dtype="float32"), positions

In [ ]:
# Training and validation tiles for both models
tiles = {}

for pair in PAIRS:
    region = regions[pair["train"]]

    train_inputs, train_targets, train_positions = make_training_tiles(
        region, FIRST_TARGET, region["train_end"], TRAIN_TILES)

    validation_inputs, validation_targets, validation_positions = make_training_tiles(
        region, region["train_end"], region["validation_end"], VALIDATION_TILES)

    tiles[pair["name"]] = {
        "train_inputs": train_inputs,
        "train_targets": train_targets,
        "train_positions": train_positions,
        "validation_inputs": validation_inputs,
        "validation_targets": validation_targets,
        "validation_positions": validation_positions,
    }

    print(pair["name"], "model - training tiles:", train_inputs.shape, " validation tiles:", validation_inputs.shape)

## 2. Model A: the CNN-LSTM forecaster

    4 past maps (2 hours), each with: water depth, past rain, local ground height
                     |
          CNN: 2 convolution layers                -> finds the spatial pattern in each map
          (the same filters for every map)            (slopes, water edges, where rain falls)
                     |
          ConvLSTM: 32 memory channels             -> goes through the maps in order and remembers
                     |                                how the water has been moving
          Convolution decoder                      -> predicted CHANGE in depth over the next 30 min
                     |
          next map = last map + predicted change   (never below 0)

**Why predict the change and not the map itself?** Over 30 minutes most of the map barely changes.
Predicting the full map would make the network re-draw everything it already knows. Predicting only the change
lets it focus on where the water moves. The last layer starts at zero, so the untrained model is exactly
"copy the last map", and training can only improve from there.

**Why ConvLSTM and not a normal LSTM?** A normal LSTM takes a list of numbers and would throw away the map's
shape. A ConvLSTM uses convolutions inside its gates, so its memory is itself a map: every cell remembers
what happened around it.

The model uses only convolutions, so its weights don't depend on the map size. It is trained on 64 x 64 tiles;
for whole maps we build the same model at the map's size and copy the trained weights into it.

In [ ]:
def build_forecaster(height=TILE_SIZE, width=TILE_SIZE):
    """CNN + ConvLSTM that predicts the water-depth map 30 minutes ahead, for maps of height x width cells."""

    inputs = keras.Input(shape=(INPUT_STEPS, height, width, CHANNELS), name="past_maps")

    # CNN: the same two convolution layers are applied to each of the 4 past maps
    x = layers.TimeDistributed(layers.Conv2D(16, 3, padding="same", activation="relu"), name="cnn_1")(inputs)
    x = layers.TimeDistributed(layers.Conv2D(16, 3, padding="same", activation="relu"), name="cnn_2")(x)

    # ConvLSTM: reads the 4 maps in order and keeps a memory map of how the water has been moving
    memory = layers.ConvLSTM2D(32, 3, padding="same", name="memory")(x)

    # Decoder: turns the memory into the predicted change in depth. It starts at zero (= copy the last map).
    x = layers.Conv2D(16, 3, padding="same", activation="relu", name="decoder")(memory)
    change = layers.Conv2D(1, 1, kernel_initializer="zeros", name="depth_change")(x)

    # Next map = last map + change, and depth can't be negative
    last_depth = inputs[:, -1, :, :, 0:1]
    next_depth = layers.ReLU(name="next_depth")(layers.Add()([last_depth, change]))

    return keras.Model(inputs, next_depth, name="cnn_lstm_forecaster")


def forecaster_for_whole_map(trained_forecaster, region):
    """Builds the forecaster at the size of a whole map and copies in the weights trained on tiles."""

    height = region["depth"].shape[1]
    width = region["depth"].shape[2]

    whole_map_forecaster = build_forecaster(height, width)
    whole_map_forecaster.set_weights(trained_forecaster.get_weights())

    return whole_map_forecaster


build_forecaster().summary()

In [ ]:
# Train one forecaster per resolution (or load the saved ones when TRAIN_NEW is False)
forecasters = {}
histories = {}

for pair in PAIRS:
    data = tiles[pair["name"]]
    weights_path = os.path.join(MODEL_DIR, "forecaster_" + pair["name"] + ".weights.h5")

    forecaster = build_forecaster()
    forecaster.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss="mse")

    if TRAIN_NEW or not os.path.exists(weights_path):
        print("Training the", pair["name"], "forecaster on", REGION_NAMES[pair["train"]])

        # Stop when the validation error hasn't improved for 4 epochs, and keep the best weights
        stop_early = keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)

        history = forecaster.fit(
            data["train_inputs"], data["train_targets"],
            validation_data=(data["validation_inputs"], data["validation_targets"]),
            epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[stop_early], verbose=2)

        histories[pair["name"]] = history.history
        forecaster.save_weights(weights_path)
    else:
        print("Loading the saved", pair["name"], "forecaster")
        forecaster.load_weights(weights_path)

    forecasters[pair["name"]] = forecaster

In [ ]:
if len(histories) > 0:
    fig, axes = plt.subplots(1, len(histories), figsize=(7 * len(histories), 4.5), layout="constrained", squeeze=False)
    fig.suptitle("Training progress: error of the forecaster on training and validation tiles", fontsize=14)

    for ax, (name, history) in zip(axes[0], histories.items()):
        epochs = np.arange(1, len(history["loss"]) + 1)

        # Square root of the mean squared error, in centimetres, so the axis is easy to read
        ax.plot(epochs, 100 * np.sqrt(history["loss"]), "o-", label="training tiles")
        ax.plot(epochs, 100 * np.sqrt(history["val_loss"]), "o-", label="validation tiles (later in time)")

        ax.set_title(name + " model")
        ax.set_xlabel("epoch (one pass over all training tiles)")
        ax.set_ylabel("typical error (cm)")
        ax.grid(alpha=0.3)
        ax.legend()

    save_figure(fig, "02_training_progress")
    plt.show()

## 3. Results of Model A on whole maps
Every forecast is compared with **"copy the last map"**: the simplest forecast possible, which says the water
will be exactly where it is now. Over 30 minutes it is already very good, so a model only counts as useful if it beats it.

| Measure | Meaning |
|---|---|
| Typical error (RMSE) | square root of the average squared error, in centimetres. Lower is better |
| **Skill** | 1 − (model's squared error / copy-the-last-map's squared error). **0** = no better than copying, **1** = perfect, **below 0** = worse than copying |
| CSI | of all cells that are flooded (> 30 cm) in the forecast or in reality, the share where both agree. 1 = perfect |

### 3a. The water-balance rule (physics check on the forecast)
In these simulations almost all of the rain stays on the ground, and only a little water crosses the edge of the map
(the dataset paper says water can leave at the edge in Australia, the UK and Mozambique, and the Indus River flows in
for Pakistan). So over the whole map, approximately:

    average rise in depth over 30 min (mm)  =  rain (mm per hour)  x  0.5 hours

With no rain, the total amount of water hardly changes; it mostly moves around.
Model A doesn't know this law, and it learned a small habit of adding water everywhere.
The rule fixes only the **total**: it shifts the predicted change so the map's average rise matches the rain.
Missing rain water is added evenly to every cell (rain falls everywhere); extra water is removed only from wet cells.
**Where** the water moves is still decided by the network.

The rule only uses rain that has **already fallen** (the last 30 minutes), so the forecast stays honest.
This is the symbolic side of the NeuroSymbolic design acting on the forecaster: a written physics law
that checks the neural network's answer. It helps Model A; for Model B (section 4) it changes almost nothing,
so it is kept as an optional physics check.

First, a check of how well the law holds in the data (the totals differ from the rain by up to about 5%):

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5), layout="constrained")
fig.suptitle("Water balance: average rise of the water over the whole map vs the rain falling at the same time", fontsize=15)

balance_rows = []

for ax, dataset in zip(axes, REGION_NAMES):
    region = regions[dataset]

    # Average depth of the whole map at every step (mm), and how much it rises in each 30 minutes
    average_depth_mm = 1000 * region["depth"].mean(axis=(1, 2), dtype="float64")
    rise_mm = np.diff(average_depth_mm)

    # Rain falling between each map and the next (mm per hour, averaged over the map), times 0.5 hours
    rain_now_mm = 0.5 * region["rain"][:len(rise_mm)].mean(axis=(1, 2), dtype="float64")

    # Rain of the 30 minutes before: the only rain the forecast is allowed to use
    rain_before_mm = np.concatenate([[0.0], rain_now_mm[:-1]])

    ax.scatter(rain_now_mm, rise_mm, s=10, alpha=0.6)
    top = max(rain_now_mm.max(), rise_mm.max())
    ax.plot([0, top], [0, top], "r--", label="rise = rain x 0.5 h")

    ax.set_title(REGION_NAMES[dataset])
    ax.set_xlabel("rain x 0.5 hours (mm)")
    ax.set_ylabel("average rise of the water (mm)")
    ax.grid(alpha=0.3)
    ax.legend()

    balance_rows.append({
        "Region": REGION_NAMES[dataset],
        "Total rise / total (rain x 0.5 h)": round(rise_mm.sum() / max(rain_now_mm.sum(), 1e-9), 3),
        "Match with rain at the same time (correlation)": round(np.corrcoef(rain_now_mm, rise_mm)[0, 1], 3),
        "Match with rain of the 30 min before (correlation)": round(np.corrcoef(rain_before_mm, rise_mm)[0, 1], 3),
    })

save_figure(fig, "03_water_balance_check")
plt.show()

balance_table = pd.DataFrame(balance_rows)
display(balance_table)
save_table(balance_table, "03_water_balance_check")

In [ ]:
def apply_water_balance(predicted, last, past_rain):
    """Water-balance rule: over the whole map, water can only grow by the rain that falls.
    Shifts the predicted change so the map's average rise equals 0.5 hours x the average rain of the last 30 minutes.
    'past_rain' is in mm per hour."""

    predicted_change = predicted - last

    # Average rise the rain allows, in metres (mm per hour x 0.5 hours / 1000)
    allowed_rise = 0.5 * float(np.mean(past_rain)) / 1000

    # Extra water the network added (positive) or rain water it missed (negative), summed over the whole map
    extra_water = (float(np.mean(predicted_change)) - allowed_rise) * predicted.size

    corrected = predicted.copy()

    if extra_water < 0:
        # Missing rain water: rain falls everywhere, so add it evenly to every cell
        corrected = corrected - extra_water / predicted.size
    else:
        # Extra water: remove it only from cells that have water, so dry hills stay dry
        wet = corrected > WET_DEPTH_M

        if wet.sum() > 0:
            corrected[wet] = corrected[wet] - extra_water / wet.sum()

    return np.maximum(corrected, 0)


def critical_success_index(predicted, true):
    """Share of cells flooded (> 30 cm) in the forecast or in reality where both agree."""

    predicted_flooded = predicted > FLOOD_DEPTH_M
    true_flooded = true > FLOOD_DEPTH_M

    hits = np.sum(predicted_flooded & true_flooded)
    misses = np.sum(~predicted_flooded & true_flooded)
    false_alarms = np.sum(predicted_flooded & ~true_flooded)

    # No flooded cells in the forecast or in reality: CSI is undefined, so the step is left out of the average
    if hits + misses + false_alarms == 0:
        return np.nan

    return hits / (hits + misses + false_alarms)


def evaluate_region(forecaster, region, first_target, last_target):
    """Forecasts every whole map from 'first_target' to 'last_target', with and without the water-balance rule,
    and compares both with copy-the-last-map. Also keeps the step where the water changes the most, for the figures."""

    rows = []
    example = None
    biggest_change = -1.0

    for target_step in range(first_target, last_target):
        model_input = make_input(region, target_step - 1)[None]
        predicted = forecaster(model_input, training=False).numpy()[0, :, :, 0]

        true = region["depth"][target_step]
        last = region["depth"][target_step - 1]

        # Rain of the last 30 minutes (the rain layer of the newest input map), back in mm per hour
        past_rain = model_input[0, -1, :, :, 1] * RAIN_SCALE
        corrected = apply_water_balance(predicted, last, past_rain)

        rows.append({
            "hours": target_step * STEP_HOURS,
            "model_squared_error": float(np.sum((predicted - true) ** 2)),
            "rule_squared_error": float(np.sum((corrected - true) ** 2)),
            "copy_squared_error": float(np.sum((last - true) ** 2)),
            "cells": true.size,
            "model_csi": critical_success_index(predicted, true),
            "rule_csi": critical_success_index(corrected, true),
            "copy_csi": critical_success_index(last, true),
        })

        total_change = float(np.abs(true - last).sum())

        if total_change > biggest_change:
            biggest_change = total_change
            example = {"step": target_step, "last": last, "true": true, "predicted": predicted, "corrected": corrected}

    return pd.DataFrame(rows), example

In [ ]:
# Forecast every whole map in the test parts of all four regions
evaluations = {}
examples = {}
whole_map_forecasters = {}

for pair in PAIRS:
    train_region = regions[pair["train"]]
    test_region = regions[pair["test"]]

    # The trained weights, copied into models the size of each whole map
    whole_map_forecasters[pair["train"]] = forecaster_for_whole_map(forecasters[pair["name"]], train_region)
    whole_map_forecasters[pair["test"]] = forecaster_for_whole_map(forecasters[pair["name"]], test_region)

    # Training region: only the last 15% of the flood, which the model never trained on
    evaluations[pair["train"]], examples[pair["train"]] = evaluate_region(
        whole_map_forecasters[pair["train"]], train_region, train_region["validation_end"], train_region["number_of_maps"])

    # Test region: the whole flood, never seen
    evaluations[pair["test"]], examples[pair["test"]] = evaluate_region(
        whole_map_forecasters[pair["test"]], test_region, FIRST_TARGET, test_region["number_of_maps"])

    print(pair["name"], "model - evaluated", REGION_NAMES[pair["train"]], "and", REGION_NAMES[pair["test"]])

In [ ]:
result_rows = []

for pair in PAIRS:
    for dataset, part in [(pair["train"], "unseen time (last 15%)"), (pair["test"], "never-seen region")]:
        evaluation = evaluations[dataset]
        cells = evaluation["cells"].sum()
        copy_error = evaluation["copy_squared_error"].sum()

        result_rows.append({
            "Region": REGION_NAMES[dataset],
            "Tested on": part,
            "Forecasts": len(evaluation),
            "Copy last map: error (cm)": round(100 * np.sqrt(copy_error / cells), 3),
            "CNN-LSTM: error (cm)": round(100 * np.sqrt(evaluation["model_squared_error"].sum() / cells), 3),
            "CNN-LSTM + water rule: error (cm)": round(100 * np.sqrt(evaluation["rule_squared_error"].sum() / cells), 3),
            "CNN-LSTM: skill": round(1 - evaluation["model_squared_error"].sum() / copy_error, 3),
            "CNN-LSTM + water rule: skill": round(1 - evaluation["rule_squared_error"].sum() / copy_error, 3),
            "Copy last map: CSI": round(evaluation["copy_csi"].mean(), 4),
            "CNN-LSTM + water rule: CSI": round(evaluation["rule_csi"].mean(), 4),
        })

result_table = pd.DataFrame(result_rows)
display(result_table)
save_table(result_table, "03_forecast_results")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")

labels = [region + "\n" + part for region, part in zip(result_table["Region"], result_table["Tested on"])]
positions = np.arange(len(labels))

ax.bar(positions - 0.2, result_table["CNN-LSTM: skill"], width=0.4, color="tab:blue", label="CNN-LSTM")
ax.bar(positions + 0.2, result_table["CNN-LSTM + water rule: skill"], width=0.4, color="tab:green",
       label="CNN-LSTM + water-balance rule")
ax.axhline(0, color="black", linewidth=1)

ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_title("Skill over copy-the-last-map (above 0 = better than copying, 1 = perfect)")
ax.set_ylabel("skill")
ax.grid(axis="y", alpha=0.3)
ax.legend()

save_figure(fig, "03_skill_per_region")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9), layout="constrained")
fig.suptitle("Forecast error at every step of the test period (lower is better)", fontsize=15)

for ax, dataset in zip(axes.flat, REGION_NAMES):
    evaluation = evaluations[dataset]
    hours = evaluation["hours"]

    ax.plot(hours, 100 * np.sqrt(evaluation["copy_squared_error"] / evaluation["cells"]),
            color="grey", label="copy the last map")
    ax.plot(hours, 100 * np.sqrt(evaluation["model_squared_error"] / evaluation["cells"]),
            color="tab:blue", label="CNN-LSTM")
    ax.plot(hours, 100 * np.sqrt(evaluation["rule_squared_error"] / evaluation["cells"]),
            color="tab:green", label="CNN-LSTM + water-balance rule")

    ax.set_title(REGION_NAMES[dataset])
    ax.set_xlabel("hours since the start of the flood")
    ax.set_ylabel("typical error over the map (cm)")
    ax.grid(alpha=0.3)
    ax.legend()

save_figure(fig, "03_error_over_time")
plt.show()

### Forecast maps
For each region, the step where the water changes the most. Columns 4–6 show the **change** over 30 minutes:
blue = water rising, red = water falling. A perfect model would make column 5 identical to column 4 and column 6 empty.

In [ ]:
column_titles = ["Water now", "True water in 30 min", "Predicted water in 30 min",
                 "True change", "Predicted change", "Model error (predicted − true)"]

fig, axes = plt.subplots(4, 6, figsize=(30, 21), layout="constrained")
fig.suptitle("CNN-LSTM + water-balance rule: forecasts 30 minutes ahead, at the moment of fastest change in each region", fontsize=20)

for row, dataset in enumerate(REGION_NAMES):
    example = examples[dataset]
    hours = example["step"] * STEP_HOURS

    # Same colour range for the three water maps, and for the three change maps
    top_depth = max(np.percentile(example["true"], 99), 0.1)
    true_change = example["true"] - example["last"]
    top_change = max(np.percentile(np.abs(true_change), 99.5), 0.01)

    pictures = [
        show_water(axes[row, 0], example["last"], top_depth),
        show_water(axes[row, 1], example["true"], top_depth),
        show_water(axes[row, 2], example["corrected"], top_depth),
        axes[row, 3].imshow(true_change, cmap="RdBu", vmin=-top_change, vmax=top_change),
        axes[row, 4].imshow(example["corrected"] - example["last"], cmap="RdBu", vmin=-top_change, vmax=top_change),
        axes[row, 5].imshow(example["corrected"] - example["true"], cmap="PuOr", vmin=-top_change, vmax=top_change),
    ]

    for column, picture in enumerate(pictures):
        if column < 3:
            unit = "water depth (m)"
        else:
            unit = "change in depth (m)"

        axes[row, column].set_title(REGION_NAMES[dataset] + "\n" + column_titles[column], fontsize=13)
        fig.colorbar(picture, ax=axes[row, column], label=unit, shrink=0.8)

    axes[row, 0].set_ylabel(str(hours) + " h into the flood", fontsize=13)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.legend(handles=[Patch(color="lightgrey", label="Dry (less than 1 cm of water)")],
           loc="lower center", ncol=1, fontsize=13, bbox_to_anchor=(0.5, -0.02))

save_figure(fig, "03_forecast_maps")
plt.show()

## 4. A second simple forecast, and Model B
"Copy the last map" is not the only simple forecast. A second one is the **trend forecast**:
the water keeps changing the way it changed in the last 30 minutes.

    next map = last map + beta x (last map − the map 30 minutes earlier)        (never below 0)

`beta` is one number, fitted on the training part of the training flood only (about 0.92 at 30 m and 0.95 at 480 m).
There is no network at all. It turns out to be far better than Model A (section 3).

**Model B** uses the same CNN + ConvLSTM as Model A, but it starts from the trend forecast and the network only learns a
small correction on top of it, counted in centimetres. Its last layer starts at zero, so before training Model B is
exactly the trend forecast.

Both models are loaded from Drive (`TRAIN_NEW` and `TRAIN_V2` are False), so nothing is retrained and the numbers
match the report.

In [ ]:
def fit_trend_factor(region, first_target, last_target):
    """The one number of the trend forecast: how much of the last 30-minute change continues in the next 30 minutes."""

    top = 0.0
    bottom = 0.0

    for step in range(first_target, last_target):
        last = region["depth"][step - 1]
        previous_change = (last - region["depth"][step - 2]).astype("float64")
        next_change = (region["depth"][step] - last).astype("float64")

        top += float(np.sum(previous_change * next_change))
        bottom += float(np.sum(previous_change ** 2))

    return top / bottom


def score_simple_forecasts(region, first_target, last_target, trend_factor):
    """Skill of the simple forecasts over copy-the-last-map, summed over all forecasts of a region."""

    errors = {"copy": 0.0, "trend": 0.0, "copy + rule": 0.0, "trend + rule": 0.0}

    for step in range(first_target, last_target):
        true = region["depth"][step]
        last = region["depth"][step - 1]
        change = last - region["depth"][step - 2]

        # Rain of the 30 minutes before the last map: the only rain a forecast may use
        past_rain = region["rain"][step - 2].astype("float32")

        trend = np.maximum(last + trend_factor * change, 0)

        forecasts = {
            "copy": last,
            "trend": trend,
            "copy + rule": apply_water_balance(last, last, past_rain),
            "trend + rule": apply_water_balance(trend, last, past_rain),
        }

        for name, forecast in forecasts.items():
            errors[name] += float(np.sum((forecast - true) ** 2, dtype="float64"))

    return {name: round(1 - errors[name] / errors["copy"], 3) for name in errors if name != "copy"}


# The trend number of each model, from the training part of its training flood
trend_factors = {}

for pair in PAIRS:
    train_region = regions[pair["train"]]
    trend_factors[pair["name"]] = fit_trend_factor(train_region, FIRST_TARGET, train_region["train_end"])

print("Trend numbers:", {name: round(value, 3) for name, value in trend_factors.items()})

In [ ]:
# Score the simple forecasts on exactly the same test parts as Model A
simple_rows = []

for pair in PAIRS:
    train_region = regions[pair["train"]]
    test_region = regions[pair["test"]]

    parts = [
        (pair["train"], "unseen time (last 15%)", train_region["validation_end"], train_region["number_of_maps"]),
        (pair["test"], "never-seen region", FIRST_TARGET, test_region["number_of_maps"]),
    ]

    for dataset, part, first_target, last_target in parts:
        skills = score_simple_forecasts(regions[dataset], first_target, last_target, trend_factors[pair["name"]])
        model_a_row = result_table[result_table["Region"] == REGION_NAMES[dataset]].iloc[0]

        simple_rows.append({
            "Region": REGION_NAMES[dataset],
            "Tested on": part,
            "Trend number": round(trend_factors[pair["name"]], 3),
            "Skill: trend": skills["trend"],
            "Skill: trend + rule": skills["trend + rule"],
            "Skill: copy + rule": skills["copy + rule"],
            "Skill: Model A": model_a_row["CNN-LSTM: skill"],
            "Skill: Model A + rule": model_a_row["CNN-LSTM + water rule: skill"],
        })

simple_table = pd.DataFrame(simple_rows)
display(simple_table)
save_table(simple_table, "04_simple_forecasts")

In [ ]:
# The network's correction is counted in centimetres, so the numbers it works with are not tiny
CHANGE_UNIT = 0.01


def build_forecaster_v2(trend_factor, height=TILE_SIZE, width=TILE_SIZE):
    """Model B: the same CNN + ConvLSTM as Model A, but it starts from the trend forecast
    and only learns a correction to it. The last layer starts at zero, so the untrained model IS the trend forecast."""

    inputs = keras.Input(shape=(INPUT_STEPS, height, width, CHANNELS), name="past_maps")

    x = layers.TimeDistributed(layers.Conv2D(16, 3, padding="same", activation="relu"), name="cnn_1")(inputs)
    x = layers.TimeDistributed(layers.Conv2D(16, 3, padding="same", activation="relu"), name="cnn_2")(x)
    memory = layers.ConvLSTM2D(32, 3, padding="same", name="memory")(x)
    x = layers.Conv2D(16, 3, padding="same", activation="relu", name="decoder")(memory)
    correction = layers.Conv2D(1, 1, kernel_initializer="zeros", name="correction")(x)

    # Trend forecast: the last map plus the change of the last 30 minutes, kept going (never below 0)
    last_depth = inputs[:, -1, :, :, 0:1]
    previous_depth = inputs[:, -2, :, :, 0:1]
    last_change = layers.Subtract()([last_depth, previous_depth])
    trend = layers.ReLU()(layers.Add()([last_depth, layers.Rescaling(float(trend_factor))(last_change)]))

    # Next map = trend forecast + the network's correction (in centimetres), never below 0
    next_depth = layers.ReLU(name="next_depth")(layers.Add()([trend, layers.Rescaling(CHANGE_UNIT)(correction)]))

    return keras.Model(inputs, next_depth, name="cnn_lstm_forecaster_v2")


def loss_in_cm(true, predicted):
    """Mean squared error counted in centimetres (in metres it is a very small number)."""

    return tf.reduce_mean(tf.square((true - predicted) / CHANGE_UNIT))


def v2_for_whole_map(trained_forecaster, region, trend_factor):
    """Builds Model B at the size of a whole map and copies in the weights trained on tiles."""

    whole_map_forecaster = build_forecaster_v2(trend_factor, region["depth"].shape[1], region["depth"].shape[2])
    whole_map_forecaster.set_weights(trained_forecaster.get_weights())

    return whole_map_forecaster


# Safety check: the untrained Model B must give exactly the trend forecast
check_model = build_forecaster_v2(trend_factors["30m"])
sample = tiles["30m"]["validation_inputs"][:50]

last_maps = sample[:, -1, :, :, 0:1]
previous_maps = sample[:, -2, :, :, 0:1]
expected = np.maximum(last_maps + trend_factors["30m"] * (last_maps - previous_maps), 0)
largest_difference = float(np.abs(check_model.predict(sample, verbose=0) - expected).max())

print("Parameters:", check_model.count_params(), "(Model A has 62,833)")
print("Largest difference between the untrained Model B and the trend forecast:", largest_difference)

In [ ]:
# Load (or, if nothing is saved yet, train) Model B for each map scale
v2_forecasters = {}

for pair in PAIRS:
    data = tiles[pair["name"]]
    weights_path = os.path.join(MODEL_DIR, "forecaster_v2_" + pair["name"] + ".weights.h5")

    forecaster = build_forecaster_v2(trend_factors[pair["name"]])
    forecaster.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss=loss_in_cm)

    if TRAIN_V2 or not os.path.exists(weights_path):
        print("Training Model B,", pair["name"], "on", REGION_NAMES[pair["train"]])

        # Error on the validation tiles before training = the error of the trend forecast itself
        start_loss = float(np.ravel(forecaster.evaluate(
            data["validation_inputs"], data["validation_targets"], batch_size=32, verbose=0))[0])
        start_weights = forecaster.get_weights()

        keras.utils.set_random_seed(42)
        stop_early = keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)

        history = forecaster.fit(
            data["train_inputs"], data["train_targets"],
            validation_data=(data["validation_inputs"], data["validation_targets"]),
            epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[stop_early], verbose=2)

        best_loss = min(history.history["val_loss"])

        # Training must not make things worse than the trend forecast it started from
        if best_loss >= start_loss:
            forecaster.set_weights(start_weights)

        print("Validation error (cm), trend forecast:", round(float(np.sqrt(start_loss)), 4),
              "| after training:", round(float(np.sqrt(min(best_loss, start_loss))), 4))

        forecaster.save_weights(weights_path)
    else:
        print("Loading the saved Model B,", pair["name"])
        forecaster.load_weights(weights_path)

    v2_forecasters[pair["name"]] = forecaster

In [ ]:
# Score Model B on whole maps exactly like Model A, and keep the per-step errors for the figures
v2_evaluations = {}
v2_examples = {}
trend_squared_errors = {}
v2_rows = []

for pair in PAIRS:
    train_region = regions[pair["train"]]
    test_region = regions[pair["test"]]
    trend_factor = trend_factors[pair["name"]]

    parts = [
        (pair["train"], "unseen time (last 15%)", train_region["validation_end"], train_region["number_of_maps"]),
        (pair["test"], "never-seen region", FIRST_TARGET, test_region["number_of_maps"]),
    ]

    for dataset, part, first_target, last_target in parts:
        region = regions[dataset]

        whole_map_model = v2_for_whole_map(v2_forecasters[pair["name"]], region, trend_factor)
        v2_evaluations[dataset], v2_examples[dataset] = evaluate_region(whole_map_model, region, first_target, last_target)
        del whole_map_model

        # Error of the trend forecast at every step (for the error-over-time figure)
        trend_errors = []

        for step in range(first_target, last_target):
            last = region["depth"][step - 1]
            trend = np.maximum(last + trend_factor * (last - region["depth"][step - 2]), 0)
            trend_errors.append(float(np.sum((trend - region["depth"][step]) ** 2, dtype="float64")))

        trend_squared_errors[dataset] = np.array(trend_errors)

        evaluation = v2_evaluations[dataset]
        copy_error = evaluation["copy_squared_error"].sum()
        simple_row = simple_table[simple_table["Region"] == REGION_NAMES[dataset]].iloc[0]

        v2_rows.append({
            "Region": REGION_NAMES[dataset],
            "Tested on": part,
            "Skill: trend": simple_row["Skill: trend"],
            "Skill: Model A": simple_row["Skill: Model A"],
            "Skill: Model B": round(1 - evaluation["model_squared_error"].sum() / copy_error, 3),
            "Skill: Model B + rule": round(1 - evaluation["rule_squared_error"].sum() / copy_error, 3),
        })

    print(pair["name"], "model - done")

v2_table = pd.DataFrame(v2_rows)
display(v2_table)
save_table(v2_table, "04_model_B_results")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5), layout="constrained")

columns = ["Skill: trend", "Skill: Model A", "Skill: Model B"]
labels = ["Trend forecast (one number, no network)", "Model A (starts from copying)", "Model B (starts from the trend)"]
colours = ["tab:orange", "tab:blue", "tab:green"]

positions = np.arange(len(v2_table))
bar_width = 0.26

for number, (column, label, colour) in enumerate(zip(columns, labels, colours)):
    bars = ax.bar(positions + (number - 1) * bar_width, v2_table[column].astype(float), width=bar_width,
                  color=colour, label=label)
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)

ax.axhline(0, color="black", linewidth=1)
ax.set_ylim(min(float(v2_table[columns].astype(float).to_numpy().min()) - 0.25, -0.25), 1.15)
ax.set_xticks(positions)
ax.set_xticklabels([region + "\n" + part for region, part in zip(v2_table["Region"], v2_table["Tested on"])])
ax.set_ylabel("skill (0 = same as copying the last map, 1 = perfect)")
ax.set_title("Skill over copy-the-last-map: the trend forecast against Model A and Model B", fontsize=14)
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right")

save_figure(fig, "04_skill_comparison")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9), layout="constrained")
fig.suptitle("Forecast error at every step of the test period (lower is better)", fontsize=15)

for ax, dataset in zip(axes.flat, REGION_NAMES):
    model_a = evaluations[dataset]
    model_b = v2_evaluations[dataset]
    hours = model_a["hours"]
    cells = model_a["cells"]

    ax.plot(hours, 100 * np.sqrt(model_a["copy_squared_error"] / cells), color="grey", label="copy the last map")
    ax.plot(hours, 100 * np.sqrt(model_a["model_squared_error"] / cells), color="tab:blue", label="Model A")
    ax.plot(hours, 100 * np.sqrt(trend_squared_errors[dataset] / cells), color="tab:orange", linewidth=2.5,
            label="trend forecast (no network)")
    ax.plot(hours, 100 * np.sqrt(model_b["model_squared_error"] / cells), color="tab:green", linestyle="--",
            linewidth=1.5, label="Model B")

    ax.set_title(REGION_NAMES[dataset])
    ax.set_xlabel("hours since the start of the flood")
    ax.set_ylabel("typical error over the map (cm)")
    ax.grid(alpha=0.3)
    ax.legend()

save_figure(fig, "04_error_over_time_with_trend")
plt.show()

### Where the forecasts go wrong
Error maps at the moment of fastest change. Purple = too much water forecast, orange = too little.
The two error columns share one colour scale in each row, so they can be compared directly.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(24, 21), layout="constrained")
fig.suptitle("Where the forecasts go wrong, 30 minutes ahead, at the moment of fastest change in each region", fontsize=18)

column_titles = ["Water now", "True change in 30 min", "Error: Model A", "Error: Model B"]
column_units = ["water depth (m)", "change in depth (m)", "error (m)", "error (m)"]

for row, dataset in enumerate(REGION_NAMES):
    model_a = examples[dataset]
    model_b = v2_examples[dataset]

    top_depth = max(np.percentile(model_a["true"], 99), 0.1)
    true_change = model_a["true"] - model_a["last"]
    top_change = max(np.percentile(np.abs(true_change), 99.5), 0.01)

    pictures = [
        show_water(axes[row, 0], model_a["last"], top_depth),
        axes[row, 1].imshow(true_change, cmap="RdBu", vmin=-top_change, vmax=top_change),
        axes[row, 2].imshow(model_a["predicted"] - model_a["true"], cmap="PuOr", vmin=-top_change, vmax=top_change),
        axes[row, 3].imshow(model_b["predicted"] - model_b["true"], cmap="PuOr", vmin=-top_change, vmax=top_change),
    ]

    for column, picture in enumerate(pictures):
        axes[row, column].set_title(REGION_NAMES[dataset] + "\n" + column_titles[column], fontsize=13)
        fig.colorbar(picture, ax=axes[row, column], label=column_units[column], shrink=0.8)

    axes[row, 0].set_ylabel(str(model_a["step"] * STEP_HOURS) + " h into the flood", fontsize=13)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.legend(handles=[Patch(color="lightgrey", label="Dry (less than 1 cm of water)")],
           loc="lower center", ncol=1, fontsize=13, bbox_to_anchor=(0.5, -0.02))

save_figure(fig, "04_error_maps")
plt.show()

**What this section shows.** At 30 minutes ahead the trend forecast already removes 91% to 99% of the error of copying.
Model A beat copying in three of four tests but scored far below the trend. Model B matches the trend (within 0.001)
but does not beat it: at 30 minutes there is almost nothing left for a network to gain. The fair test of the LSTM's
memory is a longer forecast (6 to 24 hours), which is future work.

## 5. Capsule network: severity in 30 minutes
**Severity label** of a 64 x 64 tile: the share of its normally-dry land (rivers and lakes excluded) under more than 30 cm of water.

| Class | Flooded share of dry land |
|---|---|
| None | below 5% |
| Minor | 5% – 15% |
| Moderate | 15% – 30% |
| Severe | above 30% |

The capsule network reuses Model A's CNN + ConvLSTM (frozen) and adds a capsule head:

    4 past maps -> CNN + ConvLSTM (frozen, from the forecaster) -> 2 convolution layers
               -> primary capsules: 256 small vectors of 8 numbers, each describing one part of the tile
               -> routing by agreement (3 rounds)
               -> 4 class capsules, one per severity class (vectors of 16 numbers)
               -> the LENGTH of each class capsule = how sure the network is about that class

**Why capsules?** A normal network ends in one number per class. A capsule ends in a *vector*: its length says
"how likely", and its direction stores *how* the class looks (where the water is, how it is spread).
In routing by agreement, each primary capsule votes for every class capsule, and votes that agree with each other get more weight.

In [ ]:
def squash(vectors):
    """Shrinks a vector's length to between 0 and 1 while keeping its direction."""

    squared_length = tf.reduce_sum(tf.square(vectors), axis=-1, keepdims=True)

    return squared_length / (1 + squared_length) * vectors / tf.sqrt(squared_length + 1e-7)


class Squash(layers.Layer):
    """Applies squash to every capsule."""

    def call(self, vectors):
        return squash(vectors)


class ClassCapsules(layers.Layer):
    """One capsule per class, filled by routing by agreement."""

    def __init__(self, number_of_classes, capsule_size, routing_rounds=3, **kwargs):
        super().__init__(**kwargs)
        self.number_of_classes = number_of_classes
        self.capsule_size = capsule_size
        self.routing_rounds = routing_rounds

    def build(self, input_shape):
        number_of_inputs = input_shape[1]
        input_size = input_shape[2]

        # One weight matrix for every (primary capsule, class capsule) pair
        self.vote_weights = self.add_weight(
            shape=(number_of_inputs, self.number_of_classes, self.capsule_size, input_size),
            initializer=keras.initializers.RandomNormal(stddev=0.05),
            name="vote_weights")

    def call(self, primary_capsules):
        # Every primary capsule (p) makes a vote for every class capsule (c)
        votes = tf.einsum("pcoi,bpi->bpco", self.vote_weights, primary_capsules)

        # At the start every primary capsule splits its vote equally between the classes
        routing_scores = tf.zeros_like(votes[..., 0])

        for round_number in range(self.routing_rounds):
            # How much each primary capsule sends to each class (adds up to 1 over the classes)
            coupling = tf.nn.softmax(routing_scores, axis=2)

            # Class capsule = weighted sum of the votes it received, squashed
            class_capsules = squash(tf.einsum("bpc,bpco->bco", coupling, votes))

            # Votes that agree with the class capsule get a higher score in the next round
            if round_number < self.routing_rounds - 1:
                routing_scores = routing_scores + tf.einsum("bpco,bco->bpc", votes, class_capsules)

        return class_capsules


class CapsuleLength(layers.Layer):
    """Length of each class capsule = how sure the network is about that class."""

    def call(self, capsules):
        return tf.sqrt(tf.reduce_sum(tf.square(capsules), axis=-1) + 1e-7)


def margin_loss(true_classes, lengths):
    """Capsule loss: the correct class capsule should be longer than 0.9, the others shorter than 0.1."""

    correct = true_classes * tf.square(tf.nn.relu(0.9 - lengths))
    wrong = 0.5 * (1 - true_classes) * tf.square(tf.nn.relu(lengths - 0.1))

    return tf.reduce_sum(correct + wrong, axis=-1)

In [ ]:
# Primary capsules: 4 types of 8 numbers at each of the 8 x 8 positions left after three strided convolutions
PRIMARY_TYPES = 4
PRIMARY_SIZE = 8
PRIMARY_GRID = TILE_SIZE // 8


def build_severity_model(forecaster):
    """Capsule network on top of the forecaster's frozen CNN + ConvLSTM.
    Returns the model, and a second model that also outputs the ConvLSTM memory (needed for Grad-CAM)."""

    # The CNN + ConvLSTM part of the forecaster, frozen so its weights don't change
    encoder = keras.Model(forecaster.inputs, forecaster.get_layer("memory").output, name="cnn_lstm_encoder")
    encoder.trainable = False

    inputs = keras.Input(shape=(INPUT_STEPS, TILE_SIZE, TILE_SIZE, CHANNELS), name="past_tiles")
    memory = encoder(inputs)

    # Shrink 64 x 64 -> 32 x 32 -> 16 x 16
    x = layers.Conv2D(32, 5, strides=2, padding="same", activation="relu")(memory)
    x = layers.Conv2D(32, 5, strides=2, padding="same", activation="relu")(x)

    # Primary capsules at 8 x 8 positions
    x = layers.Conv2D(PRIMARY_TYPES * PRIMARY_SIZE, 3, strides=2, padding="same")(x)
    x = layers.Reshape((PRIMARY_GRID * PRIMARY_GRID * PRIMARY_TYPES, PRIMARY_SIZE))(x)
    primary_capsules = Squash(name="primary_capsules")(x)

    # One class capsule per severity class
    class_capsules = ClassCapsules(len(SEVERITY_NAMES), 16, name="class_capsules")(primary_capsules)
    lengths = CapsuleLength(name="class_lengths")(class_capsules)

    model = keras.Model(inputs, lengths, name="capsule_severity")
    grad_cam_model = keras.Model(inputs, [memory, lengths])

    return model, grad_cam_model

In [ ]:
def tile_labels(region, targets, positions):
    """Severity class of every tile, from the depth map 30 minutes after its last input."""

    labels = []

    for target, (target_step, top, left) in zip(targets, positions):
        dry_land = region["dry_land"][top:top + TILE_SIZE, left:left + TILE_SIZE]
        labels.append(severity_class(flooded_share(target[:, :, 0], dry_land)))

    return np.array(labels)


# Labels for the training and validation tiles
for pair in PAIRS:
    data = tiles[pair["name"]]
    region = regions[pair["train"]]

    data["train_labels"] = tile_labels(region, data["train_targets"], data["train_positions"])
    data["validation_labels"] = tile_labels(region, data["validation_targets"], data["validation_positions"])

label_counts = pd.DataFrame({
    pair["name"] + " model, training tiles": np.bincount(tiles[pair["name"]]["train_labels"], minlength=4)
    for pair in PAIRS
}, index=SEVERITY_NAMES)

display(label_counts)

In [ ]:
# Train one capsule network per resolution
severity_models = {}
severity_grad_cam_models = {}

for pair in PAIRS:
    data = tiles[pair["name"]]
    weights_path = os.path.join(MODEL_DIR, "capsule_" + pair["name"] + ".weights.h5")

    model, grad_cam_model = build_severity_model(forecasters[pair["name"]])
    model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss=margin_loss,
                  metrics=[keras.metrics.CategoricalAccuracy(name="accuracy")])

    if TRAIN_NEW or not os.path.exists(weights_path):
        print("Training the", pair["name"], "capsule network")

        # Rare classes count more, so the network can't just answer "None" every time
        counts = np.bincount(data["train_labels"], minlength=4)
        class_weights = len(data["train_labels"]) / (4 * np.maximum(counts, 1))
        sample_weights = class_weights[data["train_labels"]]

        stop_early = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

        model.fit(
            data["train_inputs"], keras.utils.to_categorical(data["train_labels"], 4),
            sample_weight=sample_weights,
            validation_data=(data["validation_inputs"], keras.utils.to_categorical(data["validation_labels"], 4)),
            epochs=30, batch_size=32, callbacks=[stop_early], verbose=2)

        model.save_weights(weights_path)
    else:
        print("Loading the saved", pair["name"], "capsule network")
        model.load_weights(weights_path)

    severity_models[pair["name"]] = model
    severity_grad_cam_models[pair["name"]] = grad_cam_model

In [ ]:
# Test tiles from all four regions: last 15% of the training regions, whole flood of the never-seen regions
test_tiles = {}

for pair in PAIRS:
    train_region = regions[pair["train"]]
    test_region = regions[pair["test"]]

    parts = [
        (pair["train"], train_region["validation_end"], train_region["number_of_maps"]),
        (pair["test"], FIRST_TARGET, test_region["number_of_maps"]),
    ]

    for dataset, first_target, last_target in parts:
        region = regions[dataset]
        inputs, targets, positions = make_test_tiles(region, first_target, last_target, TEST_TILES)

        # Capsule network's answer, and the forecaster's depth map 30 minutes ahead (used by the rules)
        lengths = severity_models[pair["name"]].predict(inputs, batch_size=32, verbose=0)
        forecast = forecasters[pair["name"]].predict(inputs, batch_size=32, verbose=0)

        test_tiles[dataset] = {
            "inputs": inputs,
            "targets": targets,
            "positions": positions,
            "labels": tile_labels(region, targets, positions),
            "capsule_lengths": lengths,
            "capsule_classes": np.argmax(lengths, axis=1),
            "forecast": forecast,
            "model": pair["name"],
        }

        print(REGION_NAMES[dataset], "- test tiles:", len(inputs))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5.5), layout="constrained")
fig.suptitle("Capsule network on test tiles: true severity in 30 minutes vs the network's answer", fontsize=15)

for ax, dataset in zip(axes, REGION_NAMES):
    data = test_tiles[dataset]
    matrix = confusion_matrix(data["labels"], data["capsule_classes"], labels=range(4))

    ConfusionMatrixDisplay(matrix, display_labels=SEVERITY_NAMES).plot(ax=ax, colorbar=False, cmap="Blues")

    accuracy = np.mean(data["labels"] == data["capsule_classes"])
    ax.set_title(REGION_NAMES[dataset] + "\naccuracy " + str(round(100 * accuracy, 1)) + "%")
    ax.set_xlabel("capsule network's answer")
    ax.set_ylabel("true severity")

save_figure(fig, "05_capsule_confusion")
plt.show()

## 6. Grad-CAM: where do the models look?
Grad-CAM uses the ConvLSTM memory (32 maps, one per memory channel):

1. Pick one number to explain: for the forecaster, the **total predicted water** in 30 minutes; for the capsule network, the **length of the chosen class capsule**
2. Compute how much that number changes when each memory map changes (the gradient)
3. Average each gradient map into one weight per memory channel: how important that channel is
4. Heatmap = weighted sum of the 32 memory maps, keeping only the positive part

Bright areas are the places that pushed the forecast up the most.
For the big maps, Grad-CAM runs on a zoomed window around the fastest change, because a whole map doesn't fit in GPU memory.

In [ ]:
def grad_cam(grad_cam_model, model_input, pick_output):
    """Grad-CAM heatmap (0 to 1) over the ConvLSTM memory, for the number chosen by 'pick_output'."""

    with tf.GradientTape() as tape:
        memory, output = grad_cam_model(model_input, training=False)
        explained_number = pick_output(output)

    gradients = tape.gradient(explained_number, memory)

    # One importance weight per memory channel: its average gradient over the map
    channel_weights = tf.reduce_mean(gradients, axis=(1, 2))

    # Weighted sum of the memory maps, keeping only the positive part
    heatmap = tf.nn.relu(tf.reduce_sum(memory * channel_weights[:, None, None, :], axis=-1))[0].numpy()

    return heatmap / (heatmap.max() + 1e-8)


import gc

# Free memory left over from the whole-map evaluations
whole_map_forecasters = {}
gc.collect()

# Grad-CAM keeps every in-between map in GPU memory, so a whole 1073 x 1073 map doesn't fit.
# Instead we zoom into a window of at most 256 x 256 cells around the place where the water changes the most.
GRAD_CAM_WINDOW = 256

# Which trained model belongs to each region
model_of_region = {}

for pair in PAIRS:
    model_of_region[pair["train"]] = pair["name"]
    model_of_region[pair["test"]] = pair["name"]


def grad_cam_window(example):
    """Top-left corner and size of the window to explain, centred on the busiest area of true change."""

    rows, columns = example["true"].shape
    height = min(GRAD_CAM_WINDOW, rows)
    width = min(GRAD_CAM_WINDOW, columns)

    # Smooth the change map, so the centre is a busy area and not one single cell
    change = np.abs(example["true"] - example["last"])
    busy = uniform_filter(change, size=31)
    centre_row, centre_column = np.unravel_index(np.argmax(busy), busy.shape)

    # Keep the window inside the map
    top = int(np.clip(centre_row - height // 2, 0, rows - height))
    left = int(np.clip(centre_column - width // 2, 0, columns - width))

    return top, left, height, width

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 22), layout="constrained")
fig.suptitle("Grad-CAM of the forecaster: which areas drive the predicted water in 30 minutes", fontsize=17)

for row, dataset in enumerate(REGION_NAMES):
    region = regions[dataset]
    example = examples[dataset]
    top, left, height, width = grad_cam_window(example)

    # The trained weights, copied into a model the size of the window
    window_forecaster = build_forecaster(height, width)
    window_forecaster.set_weights(forecasters[model_of_region[dataset]].get_weights())
    grad_cam_model = keras.Model(window_forecaster.input,
                                 [window_forecaster.get_layer("memory").output, window_forecaster.output])

    model_input = make_input(region, example["step"] - 1, top, left, height, width)[None]
    heatmap = grad_cam(grad_cam_model, model_input, lambda forecast: tf.reduce_sum(forecast))

    # The same window cut out of the maps
    rows = slice(top, top + height)
    columns = slice(left, left + width)
    last = example["last"][rows, columns]
    predicted_change = example["predicted"][rows, columns] - last

    top_depth = max(np.percentile(example["true"][rows, columns], 99), 0.1)
    top_change = max(np.percentile(np.abs(predicted_change), 99.5), 0.01)
    window_text = " (rows " + str(top) + "–" + str(top + height) + ", columns " + str(left) + "–" + str(left + width) + ")"

    picture = show_water(axes[row, 0], last, top_depth)
    axes[row, 0].set_title(REGION_NAMES[dataset] + window_text + "\nwater now", fontsize=12)
    fig.colorbar(picture, ax=axes[row, 0], label="water depth (m)", shrink=0.8)

    picture = axes[row, 1].imshow(predicted_change, cmap="RdBu", vmin=-top_change, vmax=top_change)
    axes[row, 1].set_title(REGION_NAMES[dataset] + "\npredicted change in 30 min", fontsize=12)
    fig.colorbar(picture, ax=axes[row, 1], label="change in depth (m)", shrink=0.8)

    # Heatmap on top of the local ground height (grey), so the valleys and ridges are visible underneath
    axes[row, 2].imshow(region["local_height"][rows, columns], cmap="Greys_r", vmin=-2, vmax=2)
    picture = axes[row, 2].imshow(heatmap, cmap="inferno", alpha=0.6, vmin=0, vmax=1)
    axes[row, 2].set_title(REGION_NAMES[dataset] + "\nGrad-CAM (bright = most important)", fontsize=12)
    fig.colorbar(picture, ax=axes[row, 2], label="importance", shrink=0.8)

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

save_figure(fig, "06_grad_cam_forecaster")
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 18), layout="constrained")
fig.suptitle("Grad-CAM of the capsule network on never-seen regions: where it looks to decide the severity", fontsize=16)

row = 0

for pair in PAIRS:
    dataset = pair["test"]
    data = test_tiles[dataset]
    region = regions[dataset]

    # The 4 tiles with the most flooding in 30 minutes
    shares = [flooded_share(target[:, :, 0], region["dry_land"][top:top + TILE_SIZE, left:left + TILE_SIZE])
              for target, (step, top, left) in zip(data["targets"], data["positions"])]
    chosen = np.argsort(shares)[-4:]

    for column, tile in enumerate(chosen):
        chosen_class = data["capsule_classes"][tile]
        heatmap = grad_cam(severity_grad_cam_models[pair["name"]], data["inputs"][tile:tile + 1],
                           lambda lengths: lengths[0, chosen_class])

        picture = show_water(axes[row, column], data["inputs"][tile, -1, :, :, 0], 2.0)
        axes[row, column].set_title(REGION_NAMES[dataset] + ": water now\ntrue: " +
                                    SEVERITY_NAMES[data["labels"][tile]] + ", network: " +
                                    SEVERITY_NAMES[chosen_class], fontsize=11)

        axes[row + 1, column].imshow(data["inputs"][tile, -1, :, :, 0], cmap="Greys", vmin=0, vmax=2)
        axes[row + 1, column].imshow(heatmap, cmap="inferno", alpha=0.6, vmin=0, vmax=1)
        axes[row + 1, column].set_title("Grad-CAM for '" + SEVERITY_NAMES[chosen_class] + "'", fontsize=11)

    row += 2

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(picture, ax=axes[:, 3], label="water depth (m), top images", shrink=0.4)

save_figure(fig, "06_grad_cam_capsule")
plt.show()

## 7. NeuroSymbolic rules
The first symbolic part is the water-balance rule in section 3a, which corrects the forecaster with a physics law.
The second part works on severity: the neural part (capsule network) gives a severity class. The symbolic part is a set of written rules that use
facts measured from the tile and from the forecaster's predicted map. The rules can overrule the network,
and every final decision comes with a written reason.

| Rule | If... | Then... |
|---|---|---|
| 1. No water, no flood | less than 1% of the dry land is flooded now **and** in the forecast | severity = None |
| 2. Rising water | the forecast has at least as much flooding as now **and** more than 1 mm/h of rain fell in the last hour | severity can't be lower than the current level |
| 3. Agree with the forecast | the network's class is more than one class away from the class of the forecast map | use the forecast map's class |

Rule 3 links the two networks: the severity label is defined from the depth map, so the forecaster's predicted map
gives a second, independent answer that the capsule network has to agree with.

In [ ]:
# A cell counts as almost dry below 1% flooded; rule 2 needs more than 1 mm/h of rain
ALMOST_DRY_SHARE = 0.01
RAIN_RULE_MM_PER_HOUR = 1.0


def apply_rules(network_class, share_now, share_forecast, rain_last_hour):
    """Checks the capsule network's answer against the three rules. Returns the final class and the reason."""

    now_class = severity_class(share_now)
    forecast_class = severity_class(share_forecast)

    # Rule 1: no water, no flood
    if share_now < ALMOST_DRY_SHARE and share_forecast < ALMOST_DRY_SHARE:
        return 0, "Rule 1: under 1% of the land flooded now and in the forecast, so None"

    # Rule 2: rising water during rain can't make the situation less severe than it is now
    if share_forecast >= share_now and rain_last_hour > RAIN_RULE_MM_PER_HOUR and network_class < now_class:
        return now_class, "Rule 2: water rising during rain, so at least " + SEVERITY_NAMES[now_class] + " (the current level)"

    # Rule 3: the network and the forecast map are more than one class apart
    if abs(network_class - forecast_class) > 1:
        return forecast_class, "Rule 3: network said " + SEVERITY_NAMES[network_class] + \
               " but the forecast map says " + SEVERITY_NAMES[forecast_class] + ", so the forecast map wins"

    return network_class, "No rule needed: the network's answer agrees with the rules"

In [ ]:
# Apply the rules to every test tile
for dataset, data in test_tiles.items():
    region = regions[dataset]

    final_classes = []
    forecast_classes = []
    reasons = []

    for tile, (target_step, top, left) in enumerate(data["positions"]):
        dry_land = region["dry_land"][top:top + TILE_SIZE, left:left + TILE_SIZE]

        # Facts for the rules
        share_now = flooded_share(data["inputs"][tile, -1, :, :, 0], dry_land)
        share_forecast = flooded_share(data["forecast"][tile, :, :, 0], dry_land)
        rain_last_hour = data["inputs"][tile, -2:, :, :, 1].mean() * RAIN_SCALE

        final_class, reason = apply_rules(data["capsule_classes"][tile], share_now, share_forecast, rain_last_hour)

        final_classes.append(final_class)
        forecast_classes.append(severity_class(share_forecast))
        reasons.append(reason)

    data["final_classes"] = np.array(final_classes)
    data["forecast_classes"] = np.array(forecast_classes)
    data["reasons"] = reasons

In [ ]:
rule_rows = []

for dataset, data in test_tiles.items():
    rule_rows.append({
        "Region": REGION_NAMES[dataset],
        "Test tiles": len(data["labels"]),
        "Capsule network alone (%)": round(100 * np.mean(data["capsule_classes"] == data["labels"]), 1),
        "Class of the forecast map alone (%)": round(100 * np.mean(data["forecast_classes"] == data["labels"]), 1),
        "Capsule network + rules (%)": round(100 * np.mean(data["final_classes"] == data["labels"]), 1),
        "Decisions changed by the rules (%)": round(100 * np.mean(data["final_classes"] != data["capsule_classes"]), 1),
    })

rule_table = pd.DataFrame(rule_rows)
display(rule_table)
save_table(rule_table, "07_neurosymbolic_accuracy")

In [ ]:
# How often each rule decided the final answer
reason_rows = []

for dataset, data in test_tiles.items():
    first_words = pd.Series([reason.split(":")[0] for reason in data["reasons"]])
    counts = first_words.value_counts()

    row = {"Region": REGION_NAMES[dataset]}

    for rule in ["No rule needed", "Rule 1", "Rule 2", "Rule 3"]:
        row[rule] = int(counts.get(rule, 0))

    reason_rows.append(row)

reason_table = pd.DataFrame(reason_rows)
display(reason_table)
save_table(reason_table, "07_rules_used")

In [ ]:
# A few example decisions with their written reasons (never-seen regions)
example_rows = []

for pair in PAIRS:
    data = test_tiles[pair["test"]]

    # Prefer tiles where a rule changed the answer, then fill up with the others
    changed = np.where(data["final_classes"] != data["capsule_classes"])[0]
    unchanged = np.where(data["final_classes"] == data["capsule_classes"])[0]
    chosen = list(changed[:3]) + list(unchanged[:2])

    for tile in chosen:
        example_rows.append({
            "Region": REGION_NAMES[pair["test"]],
            "Hours into flood": data["positions"][tile][0] * STEP_HOURS,
            "True severity": SEVERITY_NAMES[data["labels"][tile]],
            "Capsule network": SEVERITY_NAMES[data["capsule_classes"][tile]],
            "Final decision": SEVERITY_NAMES[data["final_classes"][tile]],
            "Reason": data["reasons"][tile],
        })

example_table = pd.DataFrame(example_rows)

with pd.option_context("display.max_colwidth", None):
    display(example_table)

save_table(example_table, "07_example_decisions")

## 8. Where we stand

In [ ]:
summary_rows = []

for dataset in REGION_NAMES:
    forecast_row = v2_table[v2_table["Region"] == REGION_NAMES[dataset]].iloc[0]
    rule_row = rule_table[rule_table["Region"] == REGION_NAMES[dataset]].iloc[0]

    summary_rows.append({
        "Region": REGION_NAMES[dataset],
        "Tested on": forecast_row["Tested on"],
        "Skill: trend forecast": forecast_row["Skill: trend"],
        "Skill: Model A": forecast_row["Skill: Model A"],
        "Skill: Model B": forecast_row["Skill: Model B"],
        "Severity: capsule (%)": rule_row["Capsule network alone (%)"],
        "Severity: capsule + rules (%)": rule_row["Capsule network + rules (%)"],
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table)
save_table(summary_table, "08_summary")

In [ ]:
# Everything this notebook saved on Google Drive
for folder in [MODEL_DIR, FIGURE_DIR, TABLE_DIR]:
    print(folder)

    for file_name in sorted(os.listdir(folder)):
        print("    ", file_name)